In [0]:
import pyspark.sql.functions as F
df_employee = spark.read.table("silver_workforce.azure_blob_storage.employee")
df_payroll = spark.read.table("silver_workforce.azure_blob_storage.payroll")

In [0]:

df_joined = df_employee.join(df_payroll, on="employee_id", how="inner")

display(df_joined)

In [0]:
cols_to_keep = ["employee_id"]+ [col for col in df_joined.columns if col not in df_employee.columns]+ ["base_salary"]
df_filtered = df_joined.select(cols_to_keep)
display(df_filtered)

In [0]:
df_filtered = df_filtered.drop("gross_salary", "net_salary")
display(df_filtered)

In [0]:


df_filtered = df_filtered.withColumn(
    "net_salary",
    F.col("base_salary") +
    F.col("bonus") +
    F.col("overtime_pay") +
    F.col("commission") +
    F.col("allowances") -
    (
        F.col("tax_deduction") +
        F.col("social_security") +
        F.col("health_insurance") +
        F.col("retirement_contribution") +
        F.col("other_deductions")
    )
)

display(df_filtered)

In [0]:
df_filtered.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_workforce.azure_blob_storage.payroll")

In [0]:
display(spark.sql("select * from silver_workforce.azure_blob_storage.payroll"))